In [ ]:
import sys
import os

# 1. Determinar la ruta absoluta de la carpeta raíz (un nivel por encima de 'notebooks')
ruta_raiz = os.path.abspath(os.path.join('..'))

# 2. Añadir la ruta raíz a las variables de entorno para importar módulos 'src'
if ruta_raiz not in sys.path:
    sys.path.append(ruta_raiz)

# 3. Cambiar el directorio de trabajo del cuaderno a la raíz del proyecto para leer 'data/'
os.chdir(ruta_raiz)

print(f"Directorio de trabajo actual: {os.getcwd()}")

### Importar CSV categorizado manualmente

In [ ]:
import pandas as pd
from pathlib import Path
from src.document_processing import load_labeled_data

# 1. Definir la ruta donde el usuario debió guardar el CSV rellenado manualmente
nombre_archivo = "Cálculo de umbral" #Sustituir por el nombre del archivo ya etiquetado
ruta_csv_manual = Path(f"data/03_manual/{nombre_archivo}.csv")

# 2. Cargar los datos
print("--- IMPORTANDO DATOS CLASIFICADOS ---")
df_etiquetado = load_labeled_data(ruta_csv_manual)

# 3. Inspección inicial si la carga fue exitosa
if df_etiquetado is not None:
    print("--- Primeras 3 filas del DataFrame ---")
    # Mostramos los datos para que el usuario confirme visualmente que todo está en orden
    display(df_etiquetado.head(3))
else:
    print(">>> ACCIÓN REQUERIDA: Corrige el error en el archivo CSV antes de continuar.")

### Generar un sub-dataframe por cada dimensión añadida en el CSV

In [ ]:
# 4. Creación dinámica de sub-DataFrames por dimensión
print("--- GENERANDO SUB-DATAFRAMES DE VALIDACIÓN ---")

# Detectamos automáticamente todas las columnas que el usuario haya creado para clasificar
columnas_dimensiones = [col for col in df_etiquetado.columns if col.startswith("Similitud_")]

if not columnas_dimensiones:
    print("✗ Error: No se encontraron columnas que comiencen con 'Similitud_'. Verifica el CSV.")
else:
    # Usamos un diccionario para almacenar los DataFrames de forma dinámica
    diccionario_dfs_limpios = {}
    
    for col_dimension in columnas_dimensiones:
        # 1. Filtramos las columnas necesarias
        df_temp = df_etiquetado[["Texto", col_dimension]].copy()
        
        # 2. Eliminamos las filas donde el usuario no introdujo ninguna clasificación (NaN)
        df_temp = df_temp.dropna(subset=[col_dimension])
        
        # 3. Forzamos el tipo entero para evitar problemas en métricas posteriores (ej. 1.0 vs 1)
        df_temp[col_dimension] = df_temp[col_dimension].astype(int)
        
        # 4. Guardamos el DataFrame resultante en el diccionario
        diccionario_dfs_limpios[col_dimension] = df_temp
        
        # 5. Mostramos el resultado al usuario
        print(f"\nDimensión detectada: {col_dimension}")
        print(f"Párrafos clasificados válidos: {df_temp.shape[0]}")
        display(df_temp.head(3))

### Cálculo de embeddings para cada párrafo categorizado manualmente (por dimensiones)

In [ ]:
import os
from getpass import getpass
from pathlib import Path
# Importaremos la lógica desde src/embeddings.py cuando la construyamos
from src.embeddings import calcular_embeddings_dataframe 

# 5. Autenticación segura de la API
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Introduce tu OpenAI API Key (quedará oculta): ")

# 6. Cálculo iterativo de embeddings
print("--- INICIANDO CÁLCULO DE EMBEDDINGS ---")
directorio_salida = Path("data/04_embeddings")
directorio_salida.mkdir(parents=True, exist_ok=True)

# Diccionario para almacenar los DataFrames ya procesados con sus vectores
diccionario_dfs_embeddings = {}

for dimension, df_limpio in diccionario_dfs_limpios.items():
    print(f"\nProcesando: {dimension} ({len(df_limpio)} párrafos)")
    
    # Llamada a la función externa que encapsulará la lógica de la API
    df_con_embeddings = calcular_embeddings_dataframe(
        df=df_limpio, 
        columna_texto="Texto", 
        modelo="text-embedding-3-small"
    )
    
    diccionario_dfs_embeddings[dimension] = df_con_embeddings
    
    # Guardado estructurado y dinámico
    # Limpiamos el nombre de la dimensión para el archivo (ej. similitud_economía -> similitud_economia)
    nombre_archivo = f"embeddings_parrafos_umbral_{dimension.lower()}.pkl"
    ruta_guardado = directorio_salida / nombre_archivo
    
    df_con_embeddings.to_pickle(ruta_guardado)
    
    print(f"✓ Guardado en: {ruta_guardado}")

### Cálculo de la similitud entre cada párrafo y cada actividad de cada dimensión. Nos quedamos con el valor de similitud más alto obtenido por cada párrafo. Probamos diferentes valores de umbral, comparamos los resultados con la clasificación manual y obtenemos métricas de validación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
# Importaremos la lógica matemática y de visualización desde los módulos del repositorio
from src.similarity import calcular_max_similitud
from src.evaluation import evaluar_y_visualizar_doble

print("--- INICIANDO CÁLCULO DE SIMILITUD Y EVALUACIÓN DE UMBRALES ---")

directorio_actividades = Path("data/04_embeddings")
umbrales_optimos = {}

# Iteramos sobre el diccionario que ya creamos y procesamos en las celdas anteriores
for dimension, df_dimension in diccionario_dfs_embeddings.items():
    # 1. Identificar y cargar dinámicamente las actividades correspondientes
    # Extraemos el nombre base (ej: "Similitud_Economía" -> "economía")
    nombre_base = dimension.replace("Similitud_", "").lower()
    archivo_actividades = directorio_actividades / f"embeddings_actividades_{nombre_base}.pkl"
    
    if not archivo_actividades.exists():
        print(f"✗ Error: No se encuentra el archivo de actividades '{archivo_actividades.name}'.")
        print("Asegúrate de haberlo generado y guardado en la carpeta correcta.")
        continue
        
    df_actividades = pd.read_pickle(archivo_actividades)
    matriz_actividades = np.array(df_actividades["Embedding"].tolist())
    
    print(f"\n>> Procesando métricas para: {dimension}")
    
    # 2. Calcular similitud máxima contra la lista de actividades
    # Utilizamos el DataFrame que ya está en RAM, no necesitamos volver a leer el .pkl de los párrafos
    df_dimension["Score_Max"] = df_dimension["Embedding"].apply(
        lambda x: calcular_max_similitud(x, matriz_actividades)
    )
    
    # 3. Evaluación, barrido de umbrales y visualización de métricas
    y_real = df_dimension[dimension] # Esta es la columna con los 1 y 0 manuales
    y_scores = df_dimension["Score_Max"]
    
    # La función externa generará los gráficos y los imprimirá directamente en el cuaderno
    umbral_f1, umbral_f2 = evaluar_y_visualizar_doble(
        y_real=y_real, 
        y_scores=y_scores, 
        nombre_dimension=dimension
    )
    
    # 4. Almacenamiento de resultados
    umbrales_optimos[dimension] = {
        "Umbral_F1": umbral_f1,
        "Umbral_F2": umbral_f2
    }

print("\n--- RESUMEN DE UMBRALES RECOMENDADOS ---")
for dim, umbrales in umbrales_optimos.items():
    print(f"{dim} -> Equilibrio (F1): {umbrales['Umbral_F1']:.4f} | Priorizar Detección (F2): {umbrales['Umbral_F2']:.4f}")